In [2]:
import pandas as pd
import numpy as np
import h5py
from ase.data import chemical_symbols
from ase import Atoms, units
from ase.io import write,read
import os
import tqdm
import random
from rdkit import Chem

In [3]:
energ_to_match = read("../data/b_off_2.0_ood_original.xyz", ':')
data = h5py.File(f"../../train_grace/data/preprocessing/c_all.hdf5", "r")

In [4]:
# Create a dictionary mapping energy -> index in energ_to_match
# energy_to_index = {atom.get_potential_energy(): idx for idx, atom in enumerate(energ_to_match)}

# Note: if you have duplicate energies, this will only keep the last index
# If you need all indices for duplicates, use: 
from collections import defaultdict
energy_to_index = defaultdict(list)
for idx, atom in enumerate(energ_to_match):
    energy_to_index[atom.get_potential_energy()].append(idx)

for key in tqdm.tqdm(data.keys()):
    # Vectorize the energy calculation
    E_hartree = data[key]['formation_energy'][:]  # Load all at once
    E_eV = E_hartree * units.Hartree

    E_hartree_total = data[key]['dft_total_energy'][:]  # Load all at once
    E_ev_total = E_hartree_total * units.Hartree

    # Check if any match using dict lookup (also O(1))
    for i, energy in enumerate(E_eV):
        if energy in energy_to_index:
            original_idx = energy_to_index[energy]
            # print(f"Found matching energy: {energy} at HDF5 index {i}, corresponds to energ_to_match[{original_idx}]")
            energ_to_match[original_idx[0]].info['dft_total_energy'] = float(E_ev_total[i])

100%|██████████| 113986/113986 [00:53<00:00, 2118.04it/s]


In [9]:
write("../data/b_off_2.0_ood_with_dft_total_energies.xyz", energ_to_match)

In [10]:
import pickle
with open("energies.pkl", "rb") as f:
    energies = pickle.load(f)

In [22]:
energies["grace"]["2l"]["medium"]["force_mae_per_atom"]*1000

np.float64(15.219918620806173)